In [ ]:
import pandas as pd
df = pd.read_parquet("data/interim/application_train_cleaned.parquet")
df.head()

In [ ]:
# Credit Risk Portfolio Analytics

## 1. Data Loading and Initial Validation

print("数据规模：", df.shape)
print("重复行数：", df.duplicated().sum())
print("SK_ID_CURR重复数：", df["SK_ID_CURR"].duplicated().sum())
print("TARGET缺失数：", df["TARGET"].isna().sum())

df.info()

In [ ]:
target_summary = (
    df["TARGET"]
    .value_counts(dropna=False)
    .rename_axis("TARGET")
    .reset_index(name="CUSTOMER_COUNT")
)

target_summary["PERCENTAGE"] = (
    target_summary["CUSTOMER_COUNT"]
    / len(df)
    * 100
).round(2)

target_summary

In [ ]:
import matplotlib.pyplot as plt

target_labels = {
    0: "No payment difficulty",
    1: "Payment difficulty"
}

target_plot = target_summary.copy()
target_plot["TARGET_LABEL"] = target_plot["TARGET"].map(target_labels)

plt.figure(figsize=(8, 5))
bars = plt.bar(
    target_plot["TARGET_LABEL"],
    target_plot["CUSTOMER_COUNT"]
)

plt.title("Distribution of Loan Repayment Outcomes")
plt.xlabel("Customer Outcome")
plt.ylabel("Number of Applications")

for bar, percentage in zip(bars, target_plot["PERCENTAGE"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{percentage:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

plt.bar(
    target_summary["TARGET"].astype(str),
    target_summary["CUSTOMER_COUNT"]
)

plt.title("Loan Repayment Target Distribution")
plt.xlabel("TARGET")
plt.ylabel("Customer Count")

plt.show()

In [ ]:
##结论：坏客户只有约 8%数据类别不平衡，以后不能只看 Accuracy，因为全部预测为 0，也可能有 91% 以上准确率

In [ ]:
contract_summary = (
    df.groupby("NAME_CONTRACT_TYPE", observed=True)
    .agg(
        APPLICATION_COUNT=("SK_ID_CURR", "count"),
        PAYMENT_DIFFICULTY_COUNT=("TARGET", "sum"),
        PAYMENT_DIFFICULTY_RATE=("TARGET", "mean")
    )
    .reset_index()
)

contract_summary["PAYMENT_DIFFICULTY_RATE"] = (
    contract_summary["PAYMENT_DIFFICULTY_RATE"] * 100
).round(2)

contract_summary

In [ ]:
plt.figure(figsize=(8, 5))

bars = plt.bar(
    contract_summary["NAME_CONTRACT_TYPE"],
    contract_summary["PAYMENT_DIFFICULTY_RATE"]
)

plt.title("Payment Difficulty Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Payment Difficulty Rate (%)")

for bar, rate in zip(
    bars,
    contract_summary["PAYMENT_DIFFICULTY_RATE"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{rate:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
df["AGE_YEARS"].describe()

In [ ]:
#age figure
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(
    df["AGE_YEARS"],
    bins=30,
    edgecolor="black"
)

plt.title("Distribution of Customer Age")
plt.xlabel("Age (Years)")
plt.ylabel("Number of Customers")

plt.tight_layout()

plt.show()

In [ ]:
age_bins = [20, 30, 40, 50, 60, 70]

age_labels = [
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69"
]

df["AGE_GROUP"] = pd.cut(
    df["AGE_YEARS"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

In [ ]:
df[["AGE_YEARS", "AGE_GROUP"]].head(10)

In [ ]:
age_risk_summary = (
    df.groupby("AGE_GROUP", observed=True)
    .agg(
        CUSTOMER_COUNT=("SK_ID_CURR", "count"),
        PAYMENT_DIFFICULTY_COUNT=("TARGET", "sum"),
        PAYMENT_DIFFICULTY_RATE=("TARGET", "mean")
    )
    .reset_index()
)

age_risk_summary["PAYMENT_DIFFICULTY_RATE"] = (
    age_risk_summary["PAYMENT_DIFFICULTY_RATE"] * 100
).round(2)

age_risk_summary

In [ ]:
plt.figure(figsize=(9, 5))

bars = plt.bar(
    age_risk_summary["AGE_GROUP"].astype(str),
    age_risk_summary["PAYMENT_DIFFICULTY_RATE"]
)

plt.title("Payment Difficulty Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Payment Difficulty Rate (%)")

for bar, rate in zip(
    bars,
    age_risk_summary["PAYMENT_DIFFICULTY_RATE"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{rate:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
#结论：数据中年龄和付款困难率存在明显关系
#假设：年轻人的工作年限较短；收入可能较低；储蓄和资产积累较少；贷款金额与收入比例可能更高。

In [ ]:
df["AMT_INCOME_TOTAL"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
#这里看到有非常大的极值，如果直接画图会导致看不到所有数据
plt.figure(figsize=(8, 5))

plt.hist(
    df["AMT_INCOME_TOTAL"].dropna(),
    bins=50
)

plt.title("Distribution of Total Customer Income")
plt.xlabel("Total Income")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

In [ ]:
#筛选 收入范围在 99% 以内人群
income_99 = df["AMT_INCOME_TOTAL"].quantile(0.99)

income_plot = df[
    df["AMT_INCOME_TOTAL"] <= income_99
].copy()
#针对 收入范围在99%人群里的收入进行画图
plt.figure(figsize=(8, 5))

plt.hist(
    income_plot["AMT_INCOME_TOTAL"],
    bins=40
)

plt.title("Distribution of Customer Income Below the 99th Percentile")
plt.xlabel("Total Income")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

In [ ]:
#接下来：判断收入的水平对是否违约有没有影响

In [ ]:
income_bins = [
    0,
    100000,
    150000,
    200000,
    300000,
    float("inf")  #这里是指无限大,将无限大的收入放在300K 桶里
]

income_labels = [
    "<100K",
    "100K-150K",
    "150K-200K",
    "200K-300K",
    ">300K"
]

df["INCOME_GROUP"] = pd.cut(
    df["AMT_INCOME_TOTAL"],
    bins=income_bins,
    labels=income_labels,
    right=False
)

In [ ]:
df[
    [
        "AMT_INCOME_TOTAL",
        "INCOME_GROUP"
    ]
].head(10)

In [ ]:
#统计每组收入的违约率
income_risk_summary = (
    df.groupby("INCOME_GROUP", observed=True)
    .agg(
        CUSTOMER_COUNT=("SK_ID_CURR", "count"),
        PAYMENT_DIFFICULTY_COUNT=("TARGET", "sum"),
        PAYMENT_DIFFICULTY_RATE=("TARGET", "mean")
    )
    .reset_index()
)

income_risk_summary["PAYMENT_DIFFICULTY_RATE"] = (
    income_risk_summary["PAYMENT_DIFFICULTY_RATE"]
    * 100
).round(2)

income_risk_summary

In [ ]:
plt.figure(figsize=(9,5))

bars = plt.bar(
    income_risk_summary["INCOME_GROUP"].astype(str),
    income_risk_summary["PAYMENT_DIFFICULTY_RATE"]
)

plt.title("Payment Difficulty Rate by Income Group")
plt.xlabel("Income Group")
plt.ylabel("Payment Difficulty Rate (%)")

for bar, rate in zip(
    bars,
    income_risk_summary["PAYMENT_DIFFICULTY_RATE"]
):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{rate:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
#结论：只有高收入客户（>300K）的付款困难率明显更低，而中低收入客户之间的风险差异并不明显。

In [ ]:
#接下来看一下工作年限
df["EMPLOYMENT_YEARS"].describe(
    percentiles=[0.25,0.5,0.75,0.9,0.95,0.99]
)

In [ ]:
#将年龄分组
employment_bins = [
    0,
    2,
    5,
    10,
    20,
    float("inf")
]

employment_labels = [
    "0-2 Years",
    "2-5 Years",
    "5-10 Years",
    "10-20 Years",
    "20+ Years"
]

df["EMPLOYMENT_GROUP"] = pd.cut(
    df["EMPLOYMENT_YEARS"],
    bins=employment_bins,
    labels=employment_labels,
    right=False
)

In [ ]:
employment_risk_summary = (
    df.groupby("EMPLOYMENT_GROUP", observed=True)
    .agg(
        CUSTOMER_COUNT=("SK_ID_CURR", "count"),
        PAYMENT_DIFFICULTY_COUNT=("TARGET", "sum"),
        PAYMENT_DIFFICULTY_RATE=("TARGET", "mean")
    )
    .reset_index()
)

employment_risk_summary["PAYMENT_DIFFICULTY_RATE"] = (
    employment_risk_summary["PAYMENT_DIFFICULTY_RATE"]
    * 100
).round(2)

employment_risk_summary

In [ ]:
plt.figure(figsize=(9, 5))

bars = plt.bar(
    employment_risk_summary["EMPLOYMENT_GROUP"].astype(str),
    employment_risk_summary["PAYMENT_DIFFICULTY_RATE"]
)

plt.title("Payment Difficulty Rate by Employment Length")
plt.xlabel("Employment Length")
plt.ylabel("Payment Difficulty Rate (%)")

for bar, rate in zip(
    bars,
    employment_risk_summary["PAYMENT_DIFFICULTY_RATE"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{rate:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
#结论：工作年限越长，付款困难率越低 ， 两者存在明显关联

In [ ]:
##进入教育类型
df["NAME_EDUCATION_TYPE"].value_counts()

In [ ]:
#统计不同类别的违约率
education_summary = (
    df.groupby("NAME_EDUCATION_TYPE", observed=True)
    .agg(
        CUSTOMER_COUNT=("SK_ID_CURR", "count"),
        PAYMENT_DIFFICULTY_COUNT=("TARGET", "sum"),
        PAYMENT_DIFFICULTY_RATE=("TARGET", "mean")
    )
    .reset_index()
)

education_summary["PAYMENT_DIFFICULTY_RATE"] = (
    education_summary["PAYMENT_DIFFICULTY_RATE"] * 100
).round(2)

education_summary = education_summary.sort_values(
    by="PAYMENT_DIFFICULTY_RATE",
    ascending=False
)

education_summary

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    education_summary["NAME_EDUCATION_TYPE"],
    education_summary["PAYMENT_DIFFICULTY_RATE"]
)

plt.title("Payment Difficulty Rate by Education Level")
plt.xlabel("Education Level")
plt.ylabel("Payment Difficulty Rate (%)")

plt.xticks(rotation=20, ha="right")

for bar, rate in zip(
    bars,
    education_summary["PAYMENT_DIFFICULTY_RATE"]
):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{rate:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.show()

In [ ]:
#多变量交叉分析
age_income_summary = (
    df.groupby(
        ["AGE_GROUP", "INCOME_GROUP"],
        observed=True
    )
    .agg(
        CUSTOMER_COUNT=("SK_ID_CURR", "count"),
        PAYMENT_DIFFICULTY_RATE=("TARGET", "mean")
    )
    .reset_index()
)

age_income_summary["PAYMENT_DIFFICULTY_RATE"] = (
    age_income_summary["PAYMENT_DIFFICULTY_RATE"]
    * 100
).round(2)

age_income_summary

In [ ]:
#结论：观察到固定年龄，发现收入越高违约率越低，再固定收入，发现年龄越大违约率也越低

In [ ]:
#建立假设：
#1.年龄和收入都影响
#2.工作年限可能比收入更重要。
#3.高学历客户是否因为收入更高，还是学历本身降低了风险？

In [ ]:
def build_application_features(df):

    df = df.copy()

    # 把你之前对 application_train 做的全部特征工程代码粘贴到这里
    # 并把代码里的 application_train 全部改成 df

    return df